# Notebook 2 — Label Creation and Validation

## 1. Import Libraries and Load ML Table

In [1]:
import pandas as pd
from pathlib import Path

input_path = Path("../data/artifacts/ml_table.csv")

ml_table = pd.read_csv(input_path)

print("Shape:", ml_table.shape)
print(ml_table.head())

Shape: (99441, 20)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:0

In [2]:
print(ml_table[
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
].dtypes)

order_purchase_timestamp         str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


## 2. Convert Date Columns

In [3]:
data_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in data_columns:
    ml_table[col] = pd.to_datetime(ml_table[col], errors="coerce")

print(ml_table[data_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [4]:
print(
    ml_table[
        [
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ].isna().sum()
)

order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


## 3. Create Late Delivery Label

In [5]:
ml_table["late"] = pd.NA

delivery_mask = ml_table["order_delivered_customer_date"].notna()

ml_table.loc[delivery_mask, "late"] = (
    ml_table.loc[delivery_mask, "order_delivered_customer_date"]
    > ml_table.loc[delivery_mask, "order_estimated_delivery_date"]
).astype(int)

ml_table["late"] = ml_table["late"].astype("Int64")

## 4. Label Distribution

In [6]:
print("Label distribution:")
print(ml_table["late"].value_counts(dropna=False))

print("\nOn-time:", (ml_table["late"] == 0).sum())
print("Late:", (ml_table["late"] == 1).sum())
print("Unknown:", ml_table["late"].isna().sum())

Label distribution:
late
0       88649
1        7827
<NA>     2965
Name: count, dtype: Int64

On-time: 88649
Late: 7827
Unknown: 2965


## 5. Label Statistics

In [7]:
known_labels = ml_table["late"].dropna()

print("Known labels:", len(known_labels))
print("Late percentage:", round(known_labels.mean() * 100, 2), "%")
print("On-time percentage:", round((1 - known_labels.mean()) * 100, 2), "%")

Known labels: 96476
Late percentage: 8.11 %
On-time percentage: 91.89 %


## 6. Label Validation

In [8]:
valid_labels = ml_table.dropna(subset=["late"]).copy()

label_check = (
    valid_labels["late"]
    == (
        valid_labels["order_delivered_customer_date"]
        > valid_labels["order_estimated_delivery_date"]
    ).astype(int)
)

print("All labels correct:", label_check.all())

All labels correct: True


## 7. Unknown Label Analysis

In [9]:
unknown_status = (
    ml_table.loc[
    ml_table["late"].isna(),
    "order_status"
    ]
    .value_counts()
)

print(unknown_status)

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


## 8. Sample Label Check

In [10]:
check = (
    ml_table[
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "late"
        ]
    ]
    .dropna(subset=["late"])
    .sample(10, random_state=42)
)

check

,order_id,order_delivered_customer_date,order_estimated_delivery_date,late
22500,c58cff333993bb6b7161d7ec1350eef3,2018-04-06 02:32:49,2018-04-18,0
68942,87673b5ccb20de0a91c28cc461105d76,2018-05-23 15:28:28,2018-05-30,0
23988,80b430d0029bb33110ac31d60e87e0b8,2017-12-07 18:43:46,2017-12-27,0
31304,580603672a21252f21fa8a8b4ca85986,2018-04-26 17:44:27,2018-05-08,0
36131,c09f32e7ba9b4a134455b36eeff8fff3,2018-04-13 17:32:07,2018-04-24,0
58331,462240cb1ec4e5db517e73fff57ebfc0,2017-12-14 18:56:14,2017-12-20,0
95960,5bc2a7b8f0817443a86b69461e742cc9,2018-01-12 21:59:18,2018-02-01,0
56132,4ef8f514f95bb0a41f58965ea04e7027,2017-05-26 15:59:47,2017-06-07,0
56751,587904dc1c873ebfb6078160b4819d8e,2017-12-06 19:43:34,2017-12-15,0
92212,29c3b79aace1b72a82b1232bf494e16f,2018-04-28 15:51:50,2018-01-24,1


## 9. Saving the Labeled Table

In [11]:
from pathlib import Path

output_path = Path("../data/artifacts/labeled_table.csv")

ml_table.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print("Row:", len(ml_table))
print("Columns:", len(ml_table.columns))

Saved: ..\data\artifacts\labeled_table.csv
Row: 99441
Columns: 21
